In [13]:
import sys

if 'google.colab' in sys.modules:
  !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np

from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.core import load_exact_scores_by_match
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 35
MON_GAP_1 = -187
MON_GAP_2 = 45
HAS_BOOSTER = 1
HORIZON_NUIT = 7

# ==========================================
# 0bis. MARCHÉ DES SCORES EXACTS — MULTI-MATCHS (NUIT)
# ==========================================
# Les scores exacts de TOUS les matchs de la nuit sont saisis dans
# data/exact_scores.csv (colonnes match_id, score, cote, crowd). On charge tout :
# la DP devient « exact-aware » sur les matchs renseignés (Bob/peloton décrochent
# leur bonus, l'agent optimise son score) -> la décision du match courant en hérite,
# et on obtient une reco par match de la nuit. Cote = Pinnacle, crowd = % Winamax ;
# cote vide = score indisponible. Probas ANCRÉES sur le 1N2 du CDM_2026.csv (warning
# si écart). MATCH_DU_JOUR_ID DOIT figurer dans le CSV.
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")
print(f"📋 Matchs renseignés dans le CSV : {sorted(exact_scores_by_match)}")
print(f"   Match courant : {MATCH_DU_JOUR_ID} ({len(exact_scores_by_match.get(MATCH_DU_JOUR_ID, {}))} scores).")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
💻 Environnement Local (VS Code) détecté.
📋 Matchs renseignés dans le CSV : [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]
   Match courant : 35 (25 scores).


In [14]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE (SCORES EXACTS, MULTI-MATCHS)
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID} (DP exact-aware sur la nuit)...")

reco, wr, market_df, q_table_jour, night_markets = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10,
    exact_scores_by_match=exact_scores_by_match,   # <- DP exact-aware + reco par match
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques 1N2 synchronisées (projection).")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION MATCH COURANT ({MATCH_DU_JOUR_ID}) : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

🚀 EXÉCUTION DU PIPELINE POUR LE MATCH 35 (DP exact-aware sur la nuit)...
✅ Terminé ! 7 abaques 1N2 synchronisées (projection).

🎯 RECOMMANDATION MATCH COURANT (35) : 1-0 (Safe)
   Espérance de Victoire (WR) : 58.10%


In [15]:
# ==========================================
# 1bis. DÉTAIL DES MARCHÉS DE SCORES EXACTS (par match de la nuit)
# ==========================================
# Colonnes : E[pts MPP] (= P(outcome)*gain + P(exact)*bonus), WR base/keep/x2 (selon
# booster), WR outcome (WR si on loupe le score exact -> plancher robuste ; si WR best
# >> WR outcome, le pari ne tient que grâce au bonus, sensible au crowd Winamax).
# NB : E[pts 1/2/3] (barème simple) reste calculé dans df_m et résumé dans le Top 3
# ci-dessous, mais est retiré du tableau détaillé pour l'alléger.
#
# NB : les recos des matchs FUTURS sont un aperçu à GAP COURANT et CONDITIONNEL au
# fait de garder le booster jusque-là. Re-lance avec les gaps à jour quand le match
# devient courant.
import pandas as pd

# Noms des équipes par match (clé = position dans CDM_2026.csv + 1 = clé night_markets)
_cdm = pd.read_csv(DATA_DIR / "CDM_2026.csv")
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} - {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(_cdm.itertuples(index=False))
}


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    if HAS_BOOSTER:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)
    # E[pts 1/2/3] reste calculé dans df_m (cf. Top 3) mais retiré de l'affichage détaillé.
    view = view.drop(columns=["E[pts 1/2/3]"])

    fmt = {}
    for c in view.columns:
        if c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m*100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


for match_id, (reco_m, wr_m, df_m) in night_markets.items():
    _afficher_marche(match_id, reco_m, wr_m, df_m)


📊 MATCH 35  equateur - curacao — reco : 1-0 (Safe)  |  WR : 58.10%  ⭐ MATCH COURANT


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),12.01,16.14,50,40.36,50.71,58.10,55.20,57.47,58.10
1,4-0,1 (Dom),9.88,9.69,50,39.30,50.59,57.98,54.97,57.47,57.98
2,2-0,1 (Dom),15.91,26.24,30,39.13,50.57,57.97,54.95,57.47,57.97
3,3-0,1 (Dom),14.41,22.43,30,38.68,50.52,57.92,54.85,57.47,57.92
4,5-0,1 (Dom),5.50,2.62,70,38.21,50.47,57.87,54.71,57.47,57.87
5,2-1,1 (Dom),6.64,6.19,50,37.68,50.41,57.81,54.62,57.47,57.81
6,3-1,1 (Dom),6.02,8.20,50,37.37,50.38,57.78,54.55,57.47,57.78
7,4-1,1 (Dom),4.18,3.59,70,37.29,50.37,57.77,54.52,57.47,57.77
8,6-0,1 (Dom),2.59,1.26,70,36.17,50.25,57.66,54.28,57.47,57.66
9,5-1,1 (Dom),2.36,1.24,70,36.01,50.23,57.64,54.25,57.47,57.64


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(83.8), '2 (Ext)': np.float64(5.1), 'N (Nul)': np.float64(11.1)}
🏅 Top 3 E[pts 1/2/3] : 2-0 (1.226) | 3-0 (1.173) | 1-0 (1.160)

📊 MATCH 36  tunisie - japon — reco : 0-1 + x2  |  WR : 58.07%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-1,2 (Ext),14.69,11.58,50,68.06,50.69,57.98,58.07,57.22,58.07
1,1-2,2 (Ext),9.56,18.34,50,65.50,50.41,57.71,57.56,57.22,57.71
2,0-3,2 (Ext),8.71,11.15,50,65.07,50.36,57.66,57.47,57.22,57.66
3,0-2,2 (Ext),13.92,25.26,30,64.89,50.34,57.64,57.46,57.22,57.64
4,1-3,2 (Ext),6.07,15.56,50,63.75,50.22,57.52,57.20,57.22,57.52
5,0-4,2 (Ext),4.15,3.14,70,63.62,50.20,57.51,57.15,57.22,57.51
6,2-3,2 (Ext),2.12,3.92,70,62.20,50.04,57.36,56.87,57.22,57.36
7,1-4,2 (Ext),2.85,5.33,50,62.14,50.04,57.35,56.87,57.22,57.35
8,0-5,2 (Ext),1.57,1.52,70,61.81,50.00,57.32,56.80,57.22,57.32
9,1-5,2 (Ext),1.05,1.40,70,61.45,49.96,57.28,56.73,57.22,57.28


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(11.81), '2 (Ext)': np.float64(66.72), 'N (Nul)': np.float64(21.47)}
🏅 Top 3 E[pts 1/2/3] : 0-1 (1.078) | 1-2 (1.027) | 0-2 (1.016)

📊 MATCH 37  espagne - arabie saoudite — reco : 2-0 (Safe)  |  WR : 58.08%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,2-0,1 (Dom),16.16,14.71,50,35.67,57.18,58.08,57.66,57.24,58.08
1,1-0,1 (Dom),11.42,4.60,70,35.58,57.17,58.06,57.64,57.24,58.06
2,3-0,1 (Dom),15.82,15.18,50,35.50,57.17,58.06,57.64,57.24,58.06
3,4-0,1 (Dom),11.90,10.09,50,33.54,56.96,57.86,57.41,57.24,57.86
4,5-0,1 (Dom),7.35,5.77,50,31.27,56.72,57.62,57.14,57.24,57.62
5,2-1,1 (Dom),6.95,10.11,50,31.07,56.70,57.60,57.12,57.24,57.60
6,3-1,1 (Dom),6.59,14.69,50,30.88,56.68,57.58,57.10,57.24,57.58
7,4-1,1 (Dom),4.75,6.26,50,29.97,56.58,57.48,56.99,57.24,57.48
8,6-0,1 (Dom),2.38,4.43,70,29.25,56.51,57.41,56.90,57.24,57.41
9,5-1,1 (Dom),2.66,6.24,50,28.92,56.47,57.37,56.86,57.24,57.37


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(89.0), '2 (Ext)': np.float64(2.82), 'N (Nul)': np.float64(8.18)}
🏅 Top 3 E[pts 1/2/3] : 2-0 (1.288) | 3-0 (1.259) | 1-0 (1.199)

📊 MATCH 38  belgique - iran — reco : 1-0 (Safe)  |  WR : 58.08%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),12.54,6.80,50,32.92,57.18,58.08,57.62,57.42,58.08
1,3-0,1 (Dom),8.86,14.01,50,31.08,56.99,57.89,57.40,57.42,57.89
2,2-0,1 (Dom),12.49,21.66,30,30.40,56.92,57.82,57.33,57.42,57.82
3,3-1,1 (Dom),6.88,11.53,50,30.09,56.88,57.78,57.29,57.42,57.78
4,2-1,1 (Dom),10.22,22.45,30,29.72,56.84,57.74,57.25,57.42,57.74
5,4-1,1 (Dom),3.37,3.73,70,29.01,56.77,57.67,57.16,57.42,57.67
6,4-0,1 (Dom),4.53,5.86,50,28.92,56.76,57.66,57.15,57.42,57.66
7,0-0,N (Nul),6.12,4.04,70,31.53,56.75,57.64,57.04,57.29,57.64
8,3-2,1 (Dom),2.80,3.37,70,28.61,56.72,57.62,57.11,57.42,57.62
9,5-1,1 (Dom),1.44,1.36,70,27.66,56.62,57.53,57.00,57.42,57.53


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(68.34), '2 (Ext)': np.float64(11.92), 'N (Nul)': np.float64(19.75)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (1.068) | 2-1 (1.045) | 2-0 (1.016)


In [16]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 35 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -187,   45 | Reco : 1 (Dom) (Safe) | WR(Safe): 57.92% | WR(x2): 54.36%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -187,   45 | Reco : 1 (Dom) (Safe) | WR(Safe): 57.92% | WR(x2): 54.36%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -187,   45 | Reco : 1 (Dom) (Safe) | WR(Safe): 57.92% | WR(x2): 54.36%
------------------------------------------------------------------------------------------
▶️ MATCH 36 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -247,  -15 | Reco : 2 (Ext) (Safe) | WR(Safe): 50.85% | WR(x2): 50.16%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -187,   45 | Reco : 2 (Ext) (Safe) | WR(Safe): 57.24% | WR(x2): 56.67%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -127,  105 | Reco : 2 (Ext) (Safe) | WR(Safe): 63.65% | WR(x2): 63.05%
-----------------------------